# Alpha-Ligand — one link, one button

**You do not install anything. Not Python, not Vina, nothing.** This page is running on a free Google computer, and the single cell below installs everything it needs *on that computer* while you watch.

**How to use it:** fill in the boxes in the cell below, then press its ▶ play button **once** (or use *Runtime → Run all*). That is the whole thing. It will:

1. install the chemistry software,
2. look at your protein's pocket and work out what chemistry a binder needs,
3. search a library of millions of real published compounds for molecules that carry it,
4. dock the best ones into the pocket with AutoDock Vina and score them,
5. show you a readable results page right here and download it to your computer.

It takes roughly **5–15 minutes** and prints a running commentary with a stage header and a clock, so you can see it is working. Long silent gaps during docking are normal.

> **Want to see and control every step?** Use the
> [step-by-step notebook](https://colab.research.google.com/github/adamcantcodelol/alpha-ligand/blob/main/Alpha_Ligand_Pipeline.ipynb)
> instead. It does the same thing in separate cells, and it also has the optional
> peptide-binder branch. This one-click notebook is the small-molecule pipeline only.

---

### Before you trust any number it prints

* A Vina score is an **empirical estimate**, not a measurement. It is not a Kd, Ki or IC50.
* Scores within about **1 kcal/mol** of each other are **tied**, not ranked.
* The protein is held **rigid** and the **water is removed**. Real pockets flex.
* What you get is a **shortlist worth testing at the bench**, not a validated binder.


In [ ]:
#@title  ← Press this play button to run everything  { display-mode: "form" }

#@markdown # Alpha-Ligand — one click
#@markdown Fill in the boxes, then press the ▶ play button on the left **once**.
#@markdown It installs everything, profiles the pocket, screens a real compound
#@markdown library, docks the best molecules and shows you the results page at the
#@markdown bottom. Takes about **5–15 minutes**. You do not need to install anything.

#@markdown ---
#@markdown ### 1. Your protein
PDB_CODE = "3ERT"  #@param {type:"string"}
#@markdown Tick this to upload your own `.pdb` / `.cif` instead (a "Choose files" button appears):
UPLOAD_MY_OWN_FILE = False  #@param {type:"boolean"}

#@markdown ### 2. The pocket
#@markdown Residues lining the site you care about, as `chain:number`, comma separated.
#@markdown Leave it **empty** and the notebook will guess them from whatever is bound in the crystal.
ACTIVE_SITE_RESIDUES = "A:351,A:383,A:524"  #@param {type:"string"}
CUTOFF_ANGSTROMS = 6  #@param {type:"slider", min:4, max:10, step:0.5}

#@markdown ### 3. Where the candidate molecules come from
LIBRARY = "real compound library (slower, better)"  #@param ["real compound library (slower, better)", "starter fragments (fast)"]
NUMBER_OF_MOLECULES = 10  #@param {type:"slider", min:3, max:20, step:1}
#@markdown How hard to work. `small` is a quick test, `large` searches much more and takes much longer.
EFFORT = "medium"  #@param ["small (quickest)", "medium", "large (slowest, best)"]

#@markdown ### 4. Control ligand (a known real binder, docked as a yardstick)
CONTROL_LIGAND = "auto"  #@param ["auto", "none", "custom"]
CUSTOM_CONTROL_SMILES = ""  #@param {type:"string"}

#@markdown ---
#@markdown *Known examples to try: `3ERT` with `A:351,A:383,A:524` (oestrogen receptor + tamoxifen), or
#@markdown `5EHR` with `A:108,A:110,A:111,A:112,A:113,A:114,A:218,A:219,A:249,A:250,A:253,A:254,A:257,A:489,A:491,A:492,A:495,A:511` (SHP-2 allosteric tunnel + SHP099).*

# =============================================================================
# Everything below is machinery. You never need to read or edit it.
# =============================================================================
import os, re, sys, time, subprocess, shutil, textwrap, glob

# --- automated-test overrides (ignored in normal use) ------------------------
# The project's test harness runs this notebook on a plain kernel and sets these
# environment variables instead of clicking the form. A Colab user never has them.
_e = os.environ.get
PDB_CODE = _e("ALPHA_PDB", PDB_CODE)
ACTIVE_SITE_RESIDUES = _e("ALPHA_RESIDUES", ACTIVE_SITE_RESIDUES)
CUTOFF_ANGSTROMS = float(_e("ALPHA_CUTOFF", CUTOFF_ANGSTROMS))
LIBRARY = _e("ALPHA_LIBRARY", LIBRARY)
NUMBER_OF_MOLECULES = int(_e("ALPHA_MOLECULES", NUMBER_OF_MOLECULES))
EFFORT = _e("ALPHA_EFFORT", EFFORT)
CONTROL_LIGAND = _e("ALPHA_CONTROL", CONTROL_LIGAND)
CUSTOM_CONTROL_SMILES = _e("ALPHA_CONTROL_SMILES", CUSTOM_CONTROL_SMILES)
if _e("ALPHA_UPLOAD"):
    UPLOAD_MY_OWN_FILE = _e("ALPHA_UPLOAD") == "1"

T0 = time.time()
STAGE_T = [time.time()]


def stage(title):
    """Print a loud, numbered stage header with elapsed time so far."""
    now = time.time()
    print("\n" + "=" * 72, flush=True)
    print("%s      [%s elapsed]" % (title, _hms(now - T0)), flush=True)
    print("=" * 72, flush=True)
    STAGE_T[0] = now


def done(msg=""):
    took = time.time() - STAGE_T[0]
    print("   ...done in %s. %s" % (_hms(took), msg), flush=True)


def _hms(sec):
    sec = int(sec)
    return "%d:%02d" % (sec // 60, sec % 60)


class Friendly(Exception):
    """An error we know how to explain to a beginner."""


def bail(title, *lines):
    raise Friendly(title + "\n" + "\n".join("    " + l for l in lines))


def run(cmd, what, hints=(), quiet_ok=True):
    """Run a pipeline script, streaming its output. Raise Friendly on failure."""
    print("   $ " + " ".join(cmd[1:]) if len(cmd) > 1 else "   $ " + cmd[0], flush=True)
    proc = subprocess.Popen(cmd, stdout=subprocess.PIPE,
                            stderr=subprocess.STDOUT, text=True,
                            bufsize=1, cwd=REPO_DIR)
    tail = []
    for line in proc.stdout:
        line = line.rstrip()
        tail.append(line)
        del tail[:-40]
        print("   " + line, flush=True)
    code = proc.wait()
    if code != 0:
        blob = "\n".join(tail)
        for pattern, explanation in hints:
            if re.search(pattern, blob, re.I):
                bail("%s could not finish." % what, *explanation)
        bail("%s could not finish." % what,
             "The last thing it printed was:",
             "",
             *(tail[-8:] or ["(no output)"]),
             "",
             "If that means nothing to you, send those lines to whoever set this up.")
    return "\n".join(tail)


# -----------------------------------------------------------------------------
# Are we in Colab?
# -----------------------------------------------------------------------------
try:
    import google.colab  # noqa: F401
    IN_COLAB = True
except Exception:
    IN_COLAB = False

EFFORT_KEY = EFFORT.split()[0].lower()
PRESET = {
    "small":  dict(scan=50000,   shortlist=200,  confs=3,  exhaustiveness=4),
    "medium": dict(scan=250000,  shortlist=500,  confs=10, exhaustiveness=8),
    "large":  dict(scan=1000000, shortlist=1500, confs=16, exhaustiveness=16),
}.get(EFFORT_KEY, None)
if PRESET is None:
    PRESET = {"scan": 250000, "shortlist": 500, "confs": 10, "exhaustiveness": 8}
if _e("ALPHA_SCAN"):
    PRESET["scan"] = int(_e("ALPHA_SCAN"))

USE_REAL_LIBRARY = LIBRARY.strip().lower().startswith("real")

NATIVE_CONTROLS = {
    "3ERT": ("CC/C(=C(\\c1ccc(O)cc1)c1ccc(OCCN(C)C)cc1)c1ccccc1",
             "4-hydroxytamoxifen_(native)"),
    "5EHR": ("CC1(N)CCN(CC1)c2cnc(c(N)n2)c3cccc(Cl)c3Cl",
             "SHP099_(native_allosteric)"),
    "2SHP": ("CC1(N)CCN(CC1)c2cnc(c(N)n2)c3cccc(Cl)c3Cl",
             "SHP099_(known_allosteric_binder)"),
}

REPO_URL = "https://github.com/adamcantcodelol/alpha-ligand.git"
REPO_DIR = _e("ALPHA_REPO") or ("/content/alpha-ligand" if IN_COLAB else os.getcwd())
PY = sys.executable

OK = False
try:
    # -------------------------------------------------------------------------
    stage("STAGE 1 of 7  -  installing the software")
    # -------------------------------------------------------------------------
    if IN_COLAB:
        print("   Installing RDKit, Biopython, meeko and AutoDock Vina.")
        print("   This is the slow part (2-5 minutes) and prints a lot. That is normal.",
              flush=True)
        subprocess.run([PY, "-m", "pip", "-q", "install", "rdkit", "biopython",
                        "meeko>=0.6", "scipy", "gemmi"], check=False)
        if not shutil.which("vina"):
            subprocess.run(["wget", "-q", "-O", "/usr/local/bin/vina",
                            "https://github.com/ccsb-scripps/AutoDock-Vina/releases/"
                            "download/v1.2.5/vina_1.2.5_linux_x86_64"], check=False)
            subprocess.run(["chmod", "+x", "/usr/local/bin/vina"], check=False)
    else:
        print("   Not running in Google Colab, so nothing is installed here;")
        print("   using the Python environment that is already set up.", flush=True)

    missing = []
    for mod in ["rdkit", "Bio", "meeko", "scipy", "gemmi"]:
        try:
            __import__(mod)
        except Exception:
            missing.append(mod)
    if missing:
        bail("The software did not install properly.",
             "Missing: %s" % ", ".join(missing),
             "Press the play button again - a repeat usually fixes it.",
             "If it keeps failing, use Runtime > Restart session and run it once more.")
    if not shutil.which("vina"):
        bail("AutoDock Vina did not download.",
             "Docking cannot run without it.",
             "Press play again; if it still fails, Colab may be having network trouble.")
    done("All the chemistry software is ready.")

    # -------------------------------------------------------------------------
    stage("STAGE 2 of 7  -  getting the Alpha-Ligand code")
    # -------------------------------------------------------------------------
    if IN_COLAB:
        if os.path.isdir(os.path.join(REPO_DIR, ".git")):
            print("   Already here - updating it to the latest version.", flush=True)
            subprocess.run(["git", "-C", REPO_DIR, "pull", "--quiet"], check=False)
        else:
            subprocess.run(["git", "clone", "--quiet", REPO_URL, REPO_DIR], check=False)
    if not os.path.isfile(os.path.join(REPO_DIR, "pocket_profiler.py")):
        bail("Could not get the Alpha-Ligand code.",
             "Expected it in %s but the scripts are not there." % REPO_DIR,
             "Press play again - this is almost always a temporary network problem.")
    os.chdir(REPO_DIR)
    os.makedirs("output", exist_ok=True)
    done("Code is in %s" % REPO_DIR)

    # -------------------------------------------------------------------------
    stage("STAGE 3 of 7  -  checking what you typed")
    # -------------------------------------------------------------------------
    UPLOADED_FILE = None
    if UPLOAD_MY_OWN_FILE:
        if not IN_COLAB:
            bail("File upload only works inside Google Colab.",
                 "Untick UPLOAD_MY_OWN_FILE and type a PDB code instead.")
        from google.colab import files as _colab_files
        print("   Choose your .pdb or .cif file in the button below.", flush=True)
        _up = _colab_files.upload()
        if not _up:
            bail("You ticked the upload box but did not choose a file.",
                 "Press play again and pick a .pdb or .cif file,",
                 "or untick UPLOAD_MY_OWN_FILE and type a PDB code instead.")
        UPLOADED_FILE = os.path.abspath(list(_up.keys())[0])
        if not UPLOADED_FILE.lower().endswith((".pdb", ".cif", ".mmcif", ".ent")):
            bail("That file does not look like a protein structure.",
                 "Alpha-Ligand needs a .pdb or .cif file.",
                 "You chose: %s" % os.path.basename(UPLOADED_FILE))
        STRUCTURE = UPLOADED_FILE
        OUT_NAME = os.path.splitext(os.path.basename(UPLOADED_FILE))[0]
        print("   Uploaded:", os.path.basename(UPLOADED_FILE), flush=True)
    else:
        code = str(PDB_CODE).strip()
        if not code:
            bail("The PDB_CODE box is empty.",
                 "Type a 4-character code such as 3ERT or 5EHR,",
                 "or tick UPLOAD_MY_OWN_FILE to upload your own structure file.")
        if not re.fullmatch(r"[0-9A-Za-z]{4}", code):
            bail("'%s' is not a PDB code." % code,
                 "A PDB code is exactly 4 letters/numbers, like 3ERT, 2SHP or 5EHR.",
                 "You can look yours up at https://www.rcsb.org/",
                 "If you meant to use your own file, tick UPLOAD_MY_OWN_FILE.")
        STRUCTURE = code
        OUT_NAME = code.upper()

    # Control ligand
    CONTROL_SMILES = CONTROL_NAME = None
    if CONTROL_LIGAND == "custom":
        smi = CUSTOM_CONTROL_SMILES.strip()
        if not smi:
            bail("You chose CONTROL_LIGAND = 'custom' but left the SMILES box empty.",
                 "Either paste the SMILES of a molecule known to bind this pocket,",
                 "or set CONTROL_LIGAND back to 'auto'.")
        from rdkit import Chem as _Chem
        from rdkit import RDLogger as _RDLogger
        _RDLogger.DisableLog("rdApp.*")
        if _Chem.MolFromSmiles(smi) is None:
            bail("That control SMILES could not be read as a molecule.",
                 "You typed: %s" % smi,
                 "Check it on https://www.rcsb.org/ or your vendor page and paste it again,",
                 "or set CONTROL_LIGAND to 'auto'.")
        CONTROL_SMILES, CONTROL_NAME = smi, "custom_control"
    elif CONTROL_LIGAND == "auto" and not UPLOADED_FILE:
        if OUT_NAME in NATIVE_CONTROLS:
            CONTROL_SMILES, CONTROL_NAME = NATIVE_CONTROLS[OUT_NAME]

    # Residues: validate, or discover them
    raw = str(ACTIVE_SITE_RESIDUES).strip().strip(",")
    if raw:
        tokens = [t.strip() for t in raw.split(",") if t.strip()]
        problems = []
        for t in tokens:
            if ":" not in t:
                problems.append("'%s' has no chain letter - write it as A:%s" % (t, t))
                continue
            ch, num = (p.strip() for p in t.split(":", 1))
            if not ch:
                problems.append("'%s' is missing the chain letter before the colon "
                                "(try A:%s)" % (t, num))
            if not re.fullmatch(r"-?\d+[A-Za-z]?", num):
                problems.append("'%s' - after the colon must be a residue NUMBER, "
                                "not '%s'" % (t, num))
        if problems:
            bail("The active-site residues are not in the right format.",
                 *problems,
                 "",
                 "The format is CHAIN:NUMBER, separated by commas, e.g. A:351,A:383,A:524",
                 "Fix the box and press play again.",
                 "Or empty the box completely and this notebook will guess for you.")
        ACTIVE_SITE = ",".join(tokens)
        print("   Residues you gave: %s" % ACTIVE_SITE, flush=True)
    else:
        print("   The residues box is empty, so I will look at what is bound in", flush=True)
        print("   the crystal structure and use the residues around it.", flush=True)
        ACTIVE_SITE = None     # filled in by stage 4

    print()
    print("   ABOUT TO ANALYSE")
    print("   ----------------")
    print("   Structure     : %s" % (("your file " + os.path.basename(STRUCTURE))
                                     if UPLOADED_FILE else
                                     "PDB %s  (https://www.rcsb.org/structure/%s)"
                                     % (OUT_NAME, OUT_NAME)))
    print("   Pocket cutoff : %.1f A" % float(CUTOFF_ANGSTROMS))
    print("   Molecules     : %d   (effort: %s)" % (NUMBER_OF_MOLECULES, EFFORT_KEY))
    print("   Library       : %s" % ("ChEMBL real compounds, scanning %s rows"
                                     % f"{PRESET['scan']:,}" if USE_REAL_LIBRARY
                                     else "hand-written starter fragments"))
    print("   Control       : %s" % (CONTROL_NAME or "none (scores comparable only "
                                     "to each other)"))
    print("   Files         : output/%s_*" % OUT_NAME)
    done()

    # -------------------------------------------------------------------------
    stage("STAGE 4 of 7  -  reading the structure and profiling the pocket")
    # -------------------------------------------------------------------------
    if ACTIVE_SITE is None:
        import warnings
        warnings.filterwarnings("ignore")
        from Bio.PDB import (PDBList, PDBParser, MMCIFParser, NeighborSearch,
                             is_aa)
        IGNORED = {
            "HOH", "DOD", "WAT", "SO4", "PO4", "GOL", "EDO", "PEG", "PG4", "1PE",
            "MPD", "ACT", "FMT", "NO3", "NH4", "DMS", "TRS", "EPE", "MES", "CIT",
            "TLA", "CL", "BR", "IOD", "NA", "K", "MG", "CA", "ZN", "MN", "FE",
            "NI", "CU", "CD", "CO", "HG",
        }
        if os.path.isfile(STRUCTURE):
            parser = (MMCIFParser(QUIET=True)
                      if STRUCTURE.lower().endswith((".cif", ".mmcif"))
                      else PDBParser(QUIET=True))
            struct = parser.get_structure("user", STRUCTURE)
        else:
            os.makedirs("output/downloads", exist_ok=True)
            try:
                path = PDBList(verbose=False).retrieve_pdb_file(
                    STRUCTURE.lower(), pdir="output/downloads", file_format="mmCif")
            except Exception:
                path = None
            if not path or not os.path.isfile(path):
                bail("Could not download '%s' from the Protein Data Bank." % STRUCTURE,
                     "Either that code does not exist, or the download failed.",
                     "Check it loads at https://www.rcsb.org/structure/%s" % STRUCTURE,
                     "then press play again.")
            struct = MMCIFParser(QUIET=True).get_structure(STRUCTURE, path)
        model = next(iter(struct))
        atoms = [a for a in model.get_atoms()]
        ns = NeighborSearch(atoms)
        best = None
        for res in model.get_residues():
            het = res.id[0]
            name = res.get_resname().strip().upper()
            if het == " " or name in IGNORED:
                continue
            heavy = [a for a in res if a.element != "H"]
            if len(heavy) < 6:
                continue
            near = set()
            for a in heavy:
                for nb in ns.search(a.coord, 5.0, level="R"):
                    if is_aa(nb, standard=True):
                        near.add((nb.get_parent().id, nb.id[1]))
            if near and (best is None or len(heavy) > best[0]):
                best = (len(heavy), name, sorted(near, key=lambda x: (x[0], x[1])))
        if best is None:
            bail("I could not find anything bound inside %s to guess a pocket from."
                 % OUT_NAME,
                 "This structure appears to be empty (no drug, substrate or cofactor).",
                 "You will have to say which residues you mean. Look up the active",
                 "site on https://www.rcsb.org/structure/%s or in the paper," % OUT_NAME,
                 "then type them into ACTIVE_SITE_RESIDUES as e.g. A:351,A:383,A:524.")
        ACTIVE_SITE = ",".join("%s:%d" % (c, n) for c, n in best[2])
        print("   Found '%s' bound in the structure." % best[1])
        print("   Using the residues around it:")
        print("      %s" % ACTIVE_SITE)
        print("   (A guess from the crystal, not from biology. If you know the real")
        print("    site from a paper, type those residues in the box instead.)",
              flush=True)

    run([PY, "pocket_profiler.py", STRUCTURE, ACTIVE_SITE,
         "--cutoff", str(float(CUTOFF_ANGSTROMS))],
        "Profiling the pocket",
        hints=[
            (r"could not be downloaded|download of .* failed|neither an existing "
             r"file|not a valid 4-character",
             ["I could not download '%s' from the Protein Data Bank." % OUT_NAME,
              "Either that code does not exist or the download failed.",
              "Check it loads at https://www.rcsb.org/structure/%s, then press play"
              % OUT_NAME,
              "again. If you meant your own file, tick UPLOAD_MY_OWN_FILE."]),
            (r"chain '.*' is not in this structure|chain .* not found",
             ["One of your chain letters does not exist in this structure.",
              "You asked for: %s" % ACTIVE_SITE,
              "The line just above says which chains the structure really has",
              "(it is often just A). Fix the box and press play again."]),
            (r"residue .* was not found|not a standard amino acid|"
             r"could not read residue number",
             ["One of your residue numbers does not exist in that chain.",
              "You asked for: %s" % ACTIVE_SITE,
              "Check them on https://www.rcsb.org/structure/%s," % OUT_NAME,
              "or empty the ACTIVE_SITE_RESIDUES box and press play again to let",
              "the notebook pick residues from whatever is bound in the crystal."]),
            (r"no residues found within the cutoff",
             ["Nothing was found around the residues you gave.",
              "Slide CUTOFF_ANGSTROMS up (try 8) and press play again."]),
        ])
    pharm = "output/%s_pharmacophore.json" % OUT_NAME
    if not os.path.isfile(pharm):
        bail("The pocket profile was not written.",
             "Press play again; if it repeats, the structure may be unusual.")
    done("Pocket profiled.")

    # -------------------------------------------------------------------------
    stage("STAGE 5 of 7  -  finding candidate molecules")
    # -------------------------------------------------------------------------
    if USE_REAL_LIBRARY:
        print("   Screening ChEMBL (millions of real, published compounds).")
        print("   It prints a progress line every so often - it has NOT hung.",
              flush=True)
        cmd = [PY, "screen_library.py", pharm, "--library", "chembl",
               "--max-scan", str(PRESET["scan"]),
               "--shortlist", str(PRESET["shortlist"]),
               "--molecules", str(NUMBER_OF_MOLECULES),
               "--confs", str(PRESET["confs"])]
    else:
        print("   Assembling molecules from the hand-written starter kit.", flush=True)
        cmd = [PY, "ligand_builder.py", pharm,
               "--molecules", str(NUMBER_OF_MOLECULES),
               "--confs", str(PRESET["confs"])]
    run(cmd, "Finding candidate molecules",
        hints=[(r"network|urlopen|timed out|connection",
                ["The compound library could not be downloaded.",
                 "That is a network problem, not your input.",
                 "Press play again, or set LIBRARY to 'starter fragments (fast)',",
                 "which needs no download at all."])])
    lig_sdf = "output/%s_ligands.sdf" % OUT_NAME
    if not os.path.isfile(lig_sdf):
        bail("No candidate molecules came out of the screen.",
             "Try a larger CUTOFF_ANGSTROMS (the pocket may be tiny),",
             "or switch LIBRARY to 'starter fragments (fast)'.")
    done()

    # -------------------------------------------------------------------------
    stage("STAGE 6 of 7  -  docking them into the pocket (the slow bit)")
    # -------------------------------------------------------------------------
    print("   Vina now tries each molecule in the pocket in many orientations.")
    print("   Expect several minutes of near-silence. It has not hung.", flush=True)
    cmd = [PY, "dock_ligands.py", pharm, "--ligands", lig_sdf,
           "--receptor", STRUCTURE,
           "--exhaustiveness", str(PRESET["exhaustiveness"]), "--poses", "5"]
    if CONTROL_SMILES:
        cmd += ["--control-smiles", CONTROL_SMILES, "--control-name", CONTROL_NAME]
        print("   Control ligand: %s" % CONTROL_NAME, flush=True)
    else:
        print("   No control ligand, so the scores below can only be compared")
        print("   with each other, not with a known drug.", flush=True)
    docked_ok = True
    try:
        run(cmd, "Docking",
            hints=[(r"vina.*not found|no such file.*vina",
                    ["AutoDock Vina is missing.",
                     "Use Runtime > Restart session, then press play once more."]),
                   (r"meeko|pdbqt",
                    ["A molecule could not be prepared for docking.",
                     "Try fewer molecules, or LIBRARY = 'starter fragments (fast)'."])])
    except Friendly as exc:
        docked_ok = False
        print()
        print("   Docking failed, but everything up to here worked, so I will still")
        print("   write you a chemistry-only report. The reason docking failed:")
        print(textwrap.indent(str(exc), "   "), flush=True)
    done()

    # -------------------------------------------------------------------------
    stage("STAGE 7 of 7  -  writing the readable results page")
    # -------------------------------------------------------------------------
    run([PY, "make_report.py", pharm], "Writing the report")
    report = "output/%s_results.html" % OUT_NAME
    if not os.path.isfile(report):
        bail("The results page was not written.",
             "The raw files are still in the output folder.")
    done("Report: %s (%.0f KB)" % (report, os.path.getsize(report) / 1024))
    OK = True

except Friendly as exc:
    print()
    print("#" * 72)
    print("STOPPED - and here is what to do about it")
    print("#" * 72)
    print(str(exc))
    print()
    print("Nothing is broken. Fix the box above and press the play button again.")
    print("#" * 72)
except KeyboardInterrupt:
    print("\nStopped by you. Press play again to start over.")
except Exception as exc:                       # noqa: BLE001
    print()
    print("#" * 72)
    print("STOPPED - something unexpected went wrong")
    print("#" * 72)
    print("   %s: %s" % (type(exc).__name__, exc))
    print()
    print("   Things worth trying, in order:")
    print("     1. Press the play button again.")
    print("     2. Runtime > Restart session, then press play again.")
    print("     3. Send the lines above to whoever set this up.")
    print("#" * 72)

# -----------------------------------------------------------------------------
# Results: show the page, then download everything.
# -----------------------------------------------------------------------------
if OK:
    print("\n" + "=" * 72)
    print("FINISHED in %s. Your results page is below." % _hms(time.time() - T0))
    print("=" * 72)
    print("More negative docking scores are better. Anything within about")
    print("1 kcal/mol of another molecule is a TIE, not a ranking. These are")
    print("estimates, not measured affinities: a good score is a reason to test")
    print("a molecule at the bench, never proof that it binds.\n", flush=True)

    from IPython.display import HTML, display
    _page = open(report, encoding="utf-8").read()
    display(HTML(
        '<iframe style="width:100%; height:900px; border:1px solid #444; '
        'border-radius:8px; background:#0f1115" srcdoc="'
        + _page.replace("&", "&amp;").replace('"', "&quot;")
        + '"></iframe>'))

    wanted = [report, "output/%s_top_hits.csv" % OUT_NAME]
    wanted += [p for p in [
        "output/%s_docked.sdf" % OUT_NAME,
        "output/%s_docking_report.txt" % OUT_NAME,
        "output/%s_ligands.sdf" % OUT_NAME,
        "output/%s_ligand_report.txt" % OUT_NAME,
        "output/%s_pocket_report.txt" % OUT_NAME,
        "output/%s_pharmacophore.json" % OUT_NAME,
    ]]
    wanted = [p for p in wanted if os.path.isfile(p)]

    if IN_COLAB:
        print("Downloading %d files to your computer. Your browser will ask "
              "permission for multiple downloads - click Allow.\n" % len(wanted))
        from google.colab import files as _dl
        for p in wanted:
            print("   downloading", p)
            try:
                _dl.download(p)
            except Exception as _exc:
                print("      (could not download: %s)" % _exc)
        print("\nThe one to open is %s_results.html - double-click it in your "
              "Downloads folder." % OUT_NAME)
    else:
        print("Files written (not in Colab, so nothing was downloaded):")
        for p in wanted:
            print("   ", p)


---

## If something goes wrong

The cell tries to explain every failure in plain English and tell you which box to fix. A few common ones:

* **"is not a PDB code"** — the code must be exactly 4 characters, like `3ERT`, `2SHP`, `5EHR`. Look yours up at [rcsb.org](https://www.rcsb.org/).
* **"not in the right format"** (residues) — write them as `CHAIN:NUMBER` separated by commas, e.g. `A:351,A:383,A:524`.
* **I do not know my residues** — **leave the residues box empty**. The notebook finds whatever drug or cofactor is bound in the crystal and uses the residues around it. Check that guess against the paper before you believe it.
* **"could not find anything bound"** — that structure is empty, so there is nothing to guess from. You must supply residues yourself.
* **It stopped partway / the runtime disconnected** — Colab takes its free machines back when they sit idle. Reconnect and press play again. Nothing is lost except time.
* **Anything else** — press play again first. A repeat fixes most install hiccups.

## What you do with the files

The one to open is **`<CODE>_results.html`** — double-click it and it opens in any browser, on any computer, with nothing installed. It has the ranked molecules, pictures of them, their scores in plain language, and copyable SMILES strings you can paste into a vendor catalogue. **`<CODE>_top_hits.csv`** is the same table for Excel.

The `.sdf` files are the molecules themselves in 3D — open them next to the protein in PyMOL or the free ChimeraX if you want to look at the poses. The `.txt` and `.json` files are the raw reports behind the page.

## The other ways to run this

| | |
|---|---|
| [**This one-click notebook**](https://colab.research.google.com/github/adamcantcodelol/alpha-ligand/blob/main/Alpha_Ligand_OneClick.ipynb) | one form, one button, the whole pipeline. Start here. |
| [Step-by-step notebook](https://colab.research.google.com/github/adamcantcodelol/alpha-ligand/blob/main/Alpha_Ligand_Pipeline.ipynb) | the same pipeline in separate cells you run one at a time, plus the optional **peptide binder** branch. |
| [The web page](https://adamcantcodelol.github.io/alpha-ligand/) | pocket profiling only, instantly, with no notebook at all. |

*Running it on your own computer instead of Colab is a different thing entirely, and it does need installing Python, RDKit and Vina. Those instructions are in the repository README, under "Running it on your own computer (NOT needed for Colab)". **Ignore them while you are here.***
